# 0915 12일차

## 1. GPU 학습 환경

**GPU를 쓰는 이유**
- 딥러닝 학습은 같은 형태의 행렬 연산을 대량으로 반복함
- GPU는 이런 연산을 여러 코어에서 동시에 처리할 수 있어 병렬 연산에 유리함

**GPU 사용에 필요한 구성 요소**
1. 그래픽카드 드라이버 : 운영체제가 그래픽카드를 인식하게 해주는 프로그램
2. CUDA Toolkit : NVIDIA GPU에서 연산을 수행하게 해주는 개발 도구
3. cuDNN : CUDA 위에서 딥러닝 연산을 처리하는 라이브러리
4. TensorFlow GPU 버전 : CUDA·cuDNN을 통해 GPU로 학습하는 TensorFlow

**버전 규칙**
- 드라이버는 최신이면 됨
- CUDA·cuDNN은 TensorFlow가 만들어진 기준 버전과 맞아야 GPU를 인식함
- TensorFlow 2.11부터는 Windows에서 GPU를 지원하지 않아 2.9 버전을 따로 설치함

| 구성 요소 | TF 2.9 기준 버전 |
|---|---|
| CUDA Toolkit | 11.2.2 |
| cuDNN | 8.1.1 |
| Python | 3.10 |
| TensorFlow | `tensorflow-gpu==2.9.3` |
| numpy | 1.26.4 |

### 1-1. 가상환경 분리

기존 환경(TF 2.21)과 버전 조합이 달라 GPU용 가상환경을 따로 만듦 (conda 명령은 1일차 §1)

```bash
conda create -n tf29x-gpu python=3.10
conda activate tf29x-gpu
pip install tensorflow-gpu==2.9.3
pip install numpy==1.26.4
```

**numpy 버전을 지정하는 이유**
- `tensorflow-gpu`를 설치하면 numpy 최신 2.x가 같이 설치됨
- TF 2.9는 numpy 1.x 기준으로 만들어져 2.x에서는 import 에러가 남

### 1-2. 설치 확인

1. 드라이버 : `nvidia-smi`
   - 표에 나오는 `CUDA Version`은 이 드라이버가 지원하는 **최대** CUDA 버전이고, 설치된 버전이 아님
2. CUDA : `nvcc -V`
   - `release 11.2`처럼 실제로 설치된 CUDA 버전이 나옴
3. TensorFlow : GPU 인식 여부 (`keras35_gpu_test00`)

```python
import tensorflow as tf
print(tf.__version__)       # 2.9.3

gpus = tf.config.experimental.list_physical_devices('GPU')
print(gpus)                 # [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
```

- 빈 리스트 `[]`가 나오면 GPU를 인식하지 못해 CPU로 학습함

## 2. CPU와 GPU 실행 시간 비교

`keras35_gpu_test01` ~ `test10`

**비교 방법**
- 같은 코드를 CPU와 GPU에서 각각 실행하고 `fit` 앞뒤의 `time.time()` 차이로 학습 시간을 잼 (시간 측정은 6일차)

| 파일 | 데이터 | epochs | batch_size | CPU (초) | GPU (초) |
|---|---|---|---|---|---|
| 01 | california | 1000 | 32 | **631.20** | 995.79 |
| 02 | diabetes | 640 | 16 | 47.52 | **33.84** |
| 03 | boston | 1000 | 1 | **524.08** | 691.15 |
| 04 | dacon_ddarung | 500 | 32 | **48.31** | 131.82 |
| 05 | kaggle_bike | 200 | 8 | 513.01 | **405.62** |
| 06 | cancer | 1000 | 32 | 61.14 | **58.94** |
| 07 | santander | 100 | 32 | **620.21** | 1283.03 |
| 08 | wine | 1000 | 1 | **243.65** | 281.01 |
| 09 | fetch_covtype | 200 | 2048 | **223.85** | 372.12 |
| 10 | digits | 1000 | 1 | **1326.85** | 3481.89 |

**결과 해석**
- GPU가 빠른 경우는 diabetes·kaggle_bike·cancer 3개이고, 나머지 7개는 CPU가 빠름
- GPU로 계산하려면 데이터를 CPU 메모리에서 GPU 메모리로 옮겨야 함
- Dense 층만 쌓은 작은 모델은 GPU로 줄어드는 계산 시간보다 이 이동 시간이 더 걸려 CPU가 빠를 수 있음

## 3. 신경망 모델의 종류

데이터의 형태에 따라 쓰는 모델과 입력 차원이 다름

1. DNN (Deep Neural Network) : `Dense` 층을 쌓은 모델
   - 표 형태(csv) 데이터를 다룸
   - 입력은 2차원 `(행, 열)`
2. RNN (Recurrent Neural Network) : 순서가 있는 데이터를 다루는 모델
   - 시계열 데이터를 다룸
   - 입력은 3차원 `(행, timesteps, feature)`
   - 지금의 LLM은 RNN 다음에 나온 Transformer 구조
3. CNN (Convolutional Neural Network) : 이미지를 다루는 모델
   - 입력은 4차원 `(장수, 세로, 가로, 채널)`

## 4. CNN (합성곱 신경망)

이미지를 작은 영역씩 훑으며 특징을 뽑는 모델

DNN이 `Dense`를 쌓듯 CNN은 `Conv2D`(합성곱 층)를 쌓음

### 4-1. 이미지 데이터의 수치화

**수치화가 필요한 이유**
- 모델은 숫자만 계산할 수 있어서 이미지를 픽셀값 배열로 바꿔야 함

**이미지 데이터의 구조**
1. 픽셀값 : 8비트 이미지는 픽셀 하나가 0~255 사이의 정수
2. 채널 : 흑백은 1, 컬러(RGB)는 3
3. shape : 한 장은 3차원 `(세로, 가로, 채널)`, 여러 장을 모으면 4차원 `(장수, 세로, 가로, 채널)`

**채널 - 같은 위치에 겹쳐 쌓인 격자 수**
1. 입력 이미지 : 픽셀 하나를 표현하는 데 쓰는 숫자의 개수
   - 흑백은 밝기 하나라 1 → `[128]`
   - 컬러는 빨강·초록·파랑 세 개라 3 → `[255, 120, 60]`
   - `(224, 224, 3)`은 224×224 격자가 3장 겹쳐 있다는 뜻
2. `Conv2D`를 지난 뒤 : 필터가 만든 특징맵의 장수 (색과 무관)
   - `(10, 10, 1)` → `Conv2D(10, (3,3))` → `(8, 8, 10)`이면 특징맵 10장이 겹친 것
   - 다음 층의 필터는 그 10장을 한꺼번에 보므로 가중치가 `2×2×10`이 됨 (§4-3)

![컬러 이미지가 (224, 224, 3) 텐서로 바뀌는 과정](assets/image-to-tensor.png)

| 예시 | shape |
|---|---|
| MNIST 흑백 60000장 | `(60000, 28, 28, 1)` |
| CIFAR-10 컬러 50000장 | `(50000, 32, 32, 3)` |

### 4-2. Conv2D (합성곱 층)

이미지를 `kernel_size` 크기로 잘라 보면서, 필터를 한 칸씩 옮겨 이미지 전체를 훑는 층

```python
Conv2D(filters=10, kernel_size=(2, 2), input_shape=(28, 28, 1))
```

**주요 파라미터**
1. `kernel_size` : 한 번에 보는 영역의 크기
   - kernel은 필터 안에 들어 있는 가중치이고, `kernel_size`는 그 가중치 행렬의 크기
   - 2×2면 필터 하나에 가중치가 4개
2. `filters` : 필터의 개수
   - 필터 1개가 이미지 전체를 훑어 특징맵 1장을 만듦
   - 따라서 `filters`가 출력 채널 수가 됨

**동작 방식**
- 필터를 한 칸씩 옮기므로 잘린 영역끼리 겹침
- 겹치면서 훑기 때문에 출력의 가로·세로는 입력보다 조금 작아짐

### 4-3. Conv2D의 shape와 파라미터 계산

**계산 규칙**
1. 출력 가로·세로 = 입력 - kernel + 1
2. 출력 채널 = `filters`
3. 파라미터 개수 = (kernel 가로 × kernel 세로 × 입력 채널 + 1) × `filters`
   - +1은 필터마다 붙는 bias
   - 입력 채널이 많으면 필터 하나가 여러 장을 함께 보므로 가중치가 늘어남

**예제** (`keras36_cnn1`)

```python
model.add(Conv2D(10, (3, 3), input_shape=(10, 10, 1)))
model.add(Conv2D(5, (2, 2)))
```

![Conv2D 두 층을 지나며 바뀌는 shape와 파라미터 개수](assets/conv2d-shape-flow.svg)

### 4-4. MNIST 데이터셋

0~9 손글씨 숫자를 28×28 흑백 이미지로 모은 데이터셋 (`keras36_cnn2_mnist_imshow`)

```python
from tensorflow.keras.datasets import mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(x_train.shape, y_train.shape)   # (60000, 28, 28) (60000,)
print(x_test.shape, y_test.shape)     # (10000, 28, 28) (10000,)

print(np.unique(y_train, return_counts=True))
# (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=uint8), array([5923, 6742, 5958, 6131, 5842, 5421, 5918, 6265, 5851, 5949]))

plt.imshow(x_train[4133], 'gray')
plt.show()
```

**데이터 구조**
1. `load_data()`는 train과 test를 나눈 튜플로 반환함
2. x는 채널 차원 없이 3차원 `(장수, 28, 28)`으로 불러와짐
3. y는 0~9 정수 라벨이고, 클래스마다 약 6000개씩 있음

**주의) `pd.value_counts`**
- pandas 3.0에서 삭제되어 `AttributeError`가 남
- `pd.Series(y_test).value_counts()`처럼 Series의 메서드로 사용

### 4-5. 이미지 스케일링

**스케일링이 필요한 이유**
- 픽셀값 0~255를 그대로 넣으면 값의 크기가 커서, 스케일링으로 범위를 좁혀 줌 (스케일링의 목적은 9일차 §4)

**스케일링 방법** (`keras36_cnn3_mnist`)
1. 0~1 범위 : `x / 255.`
2. -1~1 범위 : `(x - 127.5) / 127.5`
   - 127.5는 255의 절반. 빼면 -127.5~127.5, 다시 127.5로 나누면 -1~1
   - 이미지에서 많이 사용하는 방법

```python
print(np.max(x_train), np.min(x_train))   # 255 0

x_train = x_train / 255.                  # 1.0 0.0
x_train = (x_train - 127.5) / 127.5       # 1.0 -1.0
```

**MinMaxScaler와의 차이**
- MinMaxScaler는 train의 최솟값·최댓값으로 `fit`하므로 test 값이 범위를 벗어날 수 있음
- 픽셀값은 0~255로 범위가 정해져 있어 `fit` 없이 나누기만 하면 되고, test도 범위를 벗어나지 않음